# RAG

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://openrouter.ai/
* https://openrouter.ai/mistralai/mistral-7b-instruct:free
* https://python.langchain.com/docs/integrations/providers/huggingface/
* https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


* https://docs.langchain.com/oss/python/langchain/knowledge-base
* https://docs.langchain.com/oss/python/langchain/rag#rag-agents

## Задачи для совместного разбора

In [1]:
!pip install langchain_openai==0.3.0
!pip install langchain_community==0.3.14
!pip install chromadb==0.5.0
!pip install sentence_transformers==3.3.1
!pip install langchain_huggingface==0.1.2
!pip install langchain_chroma==0.2.0
!pip install langchain_core==0.3.29
!pip install pydantic==2.10.5
!pip install onnxruntime==1.20.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 0.3.63
    Uninstalling langchain-core-0.3.63:
      Successfully uninstalled langchain-core-0.3.63
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 0.3.29 which is incompatible.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 0.3.29 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.29 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.29 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 11.9 MB/s eta 0:00:00
   ━━━━

1\. Рассмотрите процесс создания модели эмбеддингов HuggingFaceEmbeddings и его использования для получения векторных представлений текста.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("humor.txt")
documents = loader.load()

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

In [6]:
len(chunks)

42

In [7]:
chunks[:2]

[Document(metadata={'source': 'humor.txt'}, page_content='Не моё...\n\n81 год назад, 29 апреля 1945 года, советскими войсками был предпринят первый штурм рейхстага.\nВ ночь на 1 мая разгорелась яростная борьба внутри здания.\nБои шли за каждый этаж, на лестницах и в коридорах происходили рукопашные схватки...\nВ ту ночь на куполе поверженного рейхстага взвилось красное знамя, которое водрузили разведчики 756-го стрелкового полка сержант Михаил Егоров и младший сержант Мелитон Кантария во главе с заместителем командира батальона по политчасти лейтенантом Алексеем Берестом.\n\nВ 1992 году российский журналист Георгий Зотов встречался с Мелитоном Варламовичем. Отрывки из его воспоминаний об их беседе мы сегодня цитируем:\n«“Знаешь, почему я Егорову завидую? — произносит хозяин дома. — Он умер и не увидел, как распалась наша страна, за которую мы Гитлеру шею сломали. Я тоже не хотел бы до этого дожить. До сих пор не понимаю, что случилось… Скажи, сынок, куда мне знамя нести, где теперь его

In [8]:
import torch as th

embeddings = hf.embed_documents([chunk.page_content for chunk in chunks[:5]])
embeddings = th.tensor(embeddings)

2\. Рассмотрите процесс чтения документов и разбиения текста на токены

3\. Рассмотрите процесс создания vector store и поиска документов с его помощью

In [9]:
from langchain_chroma import Chroma

db = Chroma.from_documents(
    chunks,
    hf,
    persist_directory="./chroma"
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [10]:
retriever = db.as_retriever(search_kwargs={"k": 3})

In [11]:
query = "Я хочу несмешной анекдот про Кыргызстан"
retrieved = retriever.invoke(query)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [12]:
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = userdata.get('PROXYAPI_KEY')
os.environ["OPENAI_BASE_URL"] = "https://api.proxyapi.ru/openai/v1"

model = init_chat_model(
    model="gpt-4o-mini"
)

In [13]:
prompt = "Какой самый смешной анекдот про Кыргызстан (выбери из предложенных): \n\n"
for idx, doc in enumerate(retrieved):
  prompt += f"{idx}. {doc.page_content} \n"
prompt

'Какой самый смешной анекдот про Кыргызстан (выбери из предложенных): \n\n0. Не помню уже когда именно ушло это разделение, и трафик перестали делить. Но, очевидно давно, раз я даже не могу вспомнить год.\n\nГлядя на то, что происходит сейчас у нас в стране (я говорю у нас, потому что я давно уже тоже гражданка РФ), я в шоке, от того, что благодаря действиям всех этих минцифр и прочих хуеплетов, мы скатываемся к уровню Кыргызстана в те годы. Не развиваемся блять, а откатываемся, в то время, как они вообще-то развиваются ))\n\nТо ли смеяться, то ли плакать..\n5+28–Обсудить (8)ПоделитьсяИсточникMghost★★★\n1) В Индии функционирует одна из крупнейших железнодорожных сетей в мире, насчитывающая около 11 452 локомотивов.\n\n2) В некоторые периоды (например, 2015–2017 гг.) жертвами поездов становились почти 50 000 человек в год.\n\n3) В среднем речь идет о 11 000–13 000 погибших в год от поражения электрическим током. (уже поезда вырываются вперёд).\n\n4) В Индии классифицировано 3880 электро

In [14]:
model.invoke([prompt])

AIMessage(content='Предложенные вами тексты не содержат анекдотов о Кыргызстане. Если вы хотите услышать анекдот о Кыргызстане, вот один из популярных:\n\n"Почему кыргыз никогда не попадает в свой дом в темноте? \nПотому что он всегда говорит: \'А, я сейчас с торта приду!\'"\n\nЕсли вам нужно что-то другое, дайте знать!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 879, 'total_tokens': 961, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_88ed3d0b53', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ddee5-01f7-7ed3-94fc-ed2a145e63ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 879, 'output_tokens': 82, 'total_tokens': 961, 'input_token_details': {'audio': 0, '

In [22]:
!pip install langchain==1.2.15

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.3/392.3 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.2.11
    Uninstalling langsmith-0.2.11:
      Successfully uninstalled langsmith-0.2.11
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.29
    Uninstalling langchain-core-0.3.29:
      Successfully uninstalled langchain-core-0.3.29
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.25
    Uninstalling langchain-0.3.25:
      Successfully uninstalled langchain-0.3.25
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.0 requires langchain-core<0.4.0,>=0.3.29,

In [1]:
import langchain

langchain.__version__

'1.2.15'

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def extend_prompt(request: ModelRequest):
  last_query = request.state["messages"][-1]
  last_text = last_query.text
  retrieved = retriever.invoke(last_text)
  docs = "\n\n".join([d.page_content for d in retrieved])

  prompt = f"Выбери лучший анекдот: {docs}"
  return prompt


agent = create_agent(model="gpt-4o-mini", tools=[], middleware=[extend_prompt])

In [20]:
agent.invoke({"messages": ["Выбери лучший анекдот про Кыргызстан"]})

{'messages': [HumanMessage(content='Выбери лучший анекдот про Кыргызстан', additional_kwargs={}, response_metadata={}, id='b617923c-4c53-46f4-99de-63ddf7fe67be'),
  AIMessage(content='Вот анекдот, который можно считать хорошим про Кыргызстан:\n\nОдин кыргыз решил установить себе интернет. Сосед, который уже давно пользуется, говорит ему:\n— У нас, брат, интернет — это просто чудо! Ты можешь узнать любую информацию за секунду!\n— А у вас там что, прям окна в интернет?\n— Нет, у нас просто провайдер хороший!\n\nЭтот анекдот отражает особенности интернета в разных странах и может вызвать улыбку.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 847, 'total_tokens': 947, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fin

## Задачи для самостоятельного решения

In [20]:
uv pip install langchain_huggingface langchain_community langchain_chroma --no-build-isolation

Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.11 environment at: c:\Users\Rog G16\.conda\envs\FU
Resolved 102 packages in 1.90s
Uninstalled 2 packages in 1ms
Installed 4 packages in 50ms
 Downloaded kubernetes
 Downloaded grpcio
 Downloaded onnxruntime
 Downloaded chromadb
Prepared 27 packages without build isolation in 21.71s
Uninstalled 1 package in 19ms
Installed 27 packages in 216ms
 + bcrypt==5.0.0
 + build==1.5.0
 - certifi==2026.2.25
 ~ certifi==2026.4.22
 + chromadb==1.5.8
 + durationpy==0.10
 + flatbuffers==25.12.19
 + grpcio==1.80.0
 + httptools==0.7.1
 + importlib-metadata==8.7.1
 + importlib-resources==7.1.0
 + kubernetes==35.0.0
 + langchain-chroma==1.1.0
 + mmh3==5.2.1
 + oauthlib==3.3.1
 + onnxruntime==1.25.1
 + opentelemetry-api==1.41.1
 + opentelemetry-exporter-otlp-proto-common==1.41.1
 + opentelemetry-exporter-otlp-proto-grpc==1.41.1
 + opentelemetry-proto==1.41.1
 + opentelemetry-sdk==1.41.1
 + opentelemetry-semantic-conventions==0.62b1
 + overrides==7.7.0
 - protobuf==7.34.1
 + protobuf==6.33

In [10]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import SystemMessage, HumanMessage
import os
import torch as th
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1" 


<p class="task" id="1"></p>

1\. Создайте модель `HuggingFaceEmbeddings` на основе модели `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (или аналогичной, но из экосистемы huggingface).

Считайте содержимое файла `prikaz-po-VKR-bak-i-mag_izmeneniya.txt`, используя `TextLoader`. Используя `RecursiveCharacterTextSplitter`, разбейте документ на фрагменты. Важно: Установите `chunk_size` (например, 1000) и `chunk_overlap` (например, 200).

Выведите на экран количество полученных фрагментов. Выведите 3 случайных фрагмента текста на экран.
Используя созданную модель, получите эмбеддинги трех этих фрагментов в виде тензора `torch`.

- [x] Проверено на семинаре

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/prikaz-po-VKR-bak-i-mag_izmeneniya.txt")
documents = loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

In [7]:
len(chunks)

172

In [8]:
chunks[:3]

[Document(metadata={'source': 'data/prikaz-po-VKR-bak-i-mag_izmeneniya.txt'}, page_content='Федеральное государственное образовательное бюджетное учреждение высшего образования\n«Финансовый университет при Правительстве Российской Федерации»\n(Финансовый университет)\n                            ПРИКАЗ 74-1» crmx>ZJ7 202 / г.\n7\nМосква\nОб утверждении Положения о выпускной квалификационной работе по программам бакалавриата и магистратуры в Финансовом университете\n     В соответствии с Порядком проведения государственной итоговой аттестации по программам бакалавриата и магистратуры в Финансовом университете, утвержденным приказом Финуниверситета от 14.10.2016 № 1988/0, Регламентом размещения, хранения и списания курсовых проектов (работ) и выпускных квалификационных работ обучающихся в электронном виде в информационнообразовательной среде Финуниверситета, утвержденным приказом Финуниверситета от 13.09.2021 № 1853/0, Регламентом подготовки и защиты выпускной квалификационной работы, вы

In [11]:
cur_chunks = []
for idx in th.randint(0, len(chunks), (3,)):
    cur_chunks.append(chunks[idx])
    
cur_chunks

[Document(metadata={'source': 'data/prikaz-po-VKR-bak-i-mag_izmeneniya.txt'}, page_content='2 Необходимо раскрыть конкретные факты, которые заявитель считает нарушением процедуры проведения аттестационного испытания и которые повлияли на результат государственного аттестационного испытания (в случае апелляции на процедуру) либо конкретные факты, которые по мнению заявителя привели к необъективной оценке ответа на вопросы и задания билета и дополнительные вопросы (в случае апелляции на результат госэюамена).\n\t(дата)\t(подпись)\t(фамилия, инициалы)\nАпелляция полученаПриложение № 2 к приказу Финуниверситета от\nПриложение УТВЕРЖДЕН приказом Финуниверситета от 15.10.2020 № 1838/0\n                       РЕГЛАМЕНТ проведения в Финансовом университете государственной итоговой аттестации по образовательным программам бакалавриата и магистратуры с применением дистанционных образовательных технологии\n1. Общие положения'),
 Document(metadata={'source': 'data/prikaz-po-VKR-bak-i-mag_izmeneniy

In [12]:
embeddings_cur_chunks = hf.embed_documents([chunk.page_content for chunk in cur_chunks])
embeddings_cur_chunks = th.tensor(embeddings_cur_chunks)
embeddings_cur_chunks

tensor([[-0.0297,  0.0463, -0.0869,  ...,  0.1350, -0.0421,  0.0766],
        [ 0.0166,  0.0993, -0.0722,  ...,  0.0457,  0.0401,  0.0306],
        [-0.0056,  0.0510,  0.0202,  ...,  0.0520,  0.0415, -0.0126]])

<p class="task" id="2"></p>

2\. Создайте базу данных `Chroma` на основе полученных фрагментов текста, используя созданную выше модель `HuggingFaceEmbeddings`. Преобразуйте объект-хранилище в объект-retriever. При создании укажите, что по запросу требуется возращать 3 наиболее релевантных ответа. Способ поиска выберите самостоятельно.

Найдите фрагменты текста, релевантные вопросу "Сколько времени отводится на доклад?". Выведите их на экран.

- [x]  Проверено на семинаре

In [ ]:
from langchain_chroma import Chroma

db = Chroma.from_documents(
    chunks,
    hf,
    persist_directory="./chroma"
)

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
query = "Сколько времени отводится на доклад?"
retrieved = retriever.invoke(query)

In [ ]:
for i in retrieved:
    print(i.page_content)
    print('\n\n')

В заключительной части Доклада характеризуется значимость полученных результатов и Даются общие выводы.
На доклад обучающемуся отводится не более 10 минут. Для программ магистратуры Доклад Должен включать в себя: обоснование актуальности избранной темы; описание научной проблемы и формулировку цели работы; положения, ВЫНОСИМЫе на защиту; практическую значимость работы.
     В заключительной части Доклада перечисляются общие выводы и интересные результаты.
 На доклад обучающемуся отводится не более 15 минут.
2.2. Требования к презентации ВКР.
Для разработчиков методически рекомендаций:
     ДокпаД Должен сопровождаться презентацией, иллюстрирующей основные полоэюения работы с использованием мультимеДийных средств, выполненной в программе PowerPoint. Количество слаЙДОВ — 10-15.
2.3. Порядок определения результатов защиты ВКР в соответствии с пунктом
5.14 Положения.
З. Критерии оценки ВКР



2.9. В исключительных случаях изменение темы ВКР возможно не позднее, чем за 1 месяц, а уточнение 

<p class="task" id="3"></p>

3\. Используя созданный retriever, создайте RAG-агента для ответов на вопросы по документу с применением инструментов (tools).
Для этого опишите tool, который принимает в качестве аргумента вопрос, вызывает retriever для получения релевантных фрагментов документа и возвращает строку, содержащую контент этих фрагментов. Создайте агента с системным промптом, описывающим, что он должен использовать этот инструмент для поиска ответов на вопрос (не забудьте прикрепить сам инструмент). Продемонстрируйте работу агента.


- [x] Проверено на семинаре

In [ ]:
@tool
def search_in_document(query: str) -> str:
    """
    Инструмент для поиска информации в нормативных документах по ВКР.
    Принимает на вход вопрос, ищет релевантные фрагменты в тексте и возвращает их контент.
    """
    retrieved = retriever.invoke(query)
    
    docs = "\n\n".join([d.page_content for d in retrieved])
    return docs


In [ ]:
agent_with_tools = create_agent(
    model="gpt-4o-mini", 
    tools=[search_in_document]
)

messages =[
    SystemMessage(
        content=(
            """
            Ты — ИИ-ассистент учебного отдела. Твоя задача отвечать на вопросы о ВКР.
            ОБЯЗАТЕЛЬНО используй инструмент search_in_document для поиска ответа. 
            Не придумывай информацию от себя.
            """
        )
    ),
    HumanMessage(content="Сколько времени отводится на доклад?")
]

response = agent_with_tools.invoke({"messages": messages})
display(Markdown("`user`: Сколько времени отводится на доклад?"))

display(Markdown("`model`:\n" + response["messages"][-1].content))

`user`: Сколько времени отводится на доклад?

`model`:
На доклад обучающемуся отводится не более 10 минут для программ бакалавриата и не более 15 минут для программ магистратуры.

<p class="task" id="4"></p>

4\. Используя созданный retriever, создайте RAG-агента для ответов на вопросы по документу с применением `dynamic_prompt`.

Для этого опишите функцию, которая получает текст вопроса, вызывает retriever для получения релевантных фрагментов документа и возвращает строку, содержащую контент этих фрагментов и указание использовать их для ответа на вопрос.

Создайте агента с middleware для создания динамического промпта. Продемонстрируйте работу агента.


- [x] Проверено на семинаре

In [ ]:
@dynamic_prompt
def extend_prompt_with_rag(request: ModelRequest):
    last_query = request.state["messages"][-1]
    
    last_text = last_query.text 
    
    retrieved = retriever.invoke(last_text)
    docs = "\n\n".join([d.page_content for d in retrieved])

    prompt = (
        "Ты — ИИ-ассистент учебного отдела.\n"
        "Ответь на вопрос студента, опираясь ТОЛЬКО на следующий текст документа:\n\n"
        f"Текст документа:\n{docs}\n\n"
        f"Вопрос студента: {last_text}"
    )
    return prompt

dynamic_agent = create_agent(
    model="gpt-4o-mini", 
    tools=[], 
    middleware=[extend_prompt_with_rag]
)

result = dynamic_agent.invoke({
    "messages": [HumanMessage(content="Сколько времени отводится на доклад?")]
})
display(Markdown("`user`: Сколько времени отводится на доклад?"))


display(Markdown("`model`:\n" + result["messages"][-1].content))

`user`: Сколько времени отводится на доклад?

`model`:
На доклад обучающемуся отводится не более 10 минут для программ бакалавриата и не более 15 минут для программ магистратуры.